# DATA 612 — iTransformer on Criteo (12h)

Comparison Transformer from the proposal: **iTransformer** (Liu et al., *iTransformer: Inverted Transformers Are
Effective for Time Series Forecasting*, ICLR 2024), implemented as proposed in `docs/design_notes.md` §3.

**Architecture (inverted: attention across variables, not across time)**
- **One token per variable**: the whole `LOOKBACK`-window history of that variable is embedded as one token
  (`Linear(LOOKBACK → d)`), plus a learned variable-identity embedding (many sparse series are all zero).
- **Variables**: the `TOP_CAMPAIGNS` most frequent campaigns in the training sample (each campaign's per-window
  log-impressions), one **"other campaigns"** token, and the 7 activity series (impressions, clicks, campaigns,
  cost, past conversions, active, has-record) → 108 tokens with the defaults.
- **Attention across tokens** models cross-campaign effects — the research question the design notes point to.
- **Heads**: conversion from mean+max pooled tokens plus the same static features as the other models (incl. the
  full-history features). Campaigns: a pooled linear head over all campaigns, **plus a per-token score for each
  top campaign** (its own token predicts its own next-window appearance), plus the same history term as the other
  Transformers (`COPY_HISTORY`).

**Identical to the other notebooks:** data, full-history features (version 2), splits, training sample,
probability correction, F1 threshold from validation, campaign evaluation sample, and the repo's `metrics.py`.

**Runtime:** GPU (A100 or L4) **with High-RAM on**. When finished: Runtime → *Disconnect and delete runtime*.

## 1. Settings

In [1]:
import os, sys, json, time, math, subprocess, importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/DATA612_data")
    REPO_DIR  = Path("/content/repo")
else:
    DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", "drive_sim/DATA612_data")).resolve()
    REPO_DIR  = Path(os.environ.get("REPO_DIR", "repo")).resolve()

DATASET    = "criteo_w12h"          # "criteo_w12h_smoke" for a quick test run
PROTOCOLS  = ["chronological", "user_disjoint"]       # also available: "user_disjoint_future"
USE_HISTORY_FEATURES = True         # the repo's full-history features (src/data/features.py) as extra inputs
HIST_DROP  = ["day_index"]          # absolute day: unseen in the chronological test period, so left out
TOP_CAMPAIGNS = 100                 # campaigns with their own token (the rest share one "other" token)
HEADS      = [4]
SEEDS      = [0]                    # final table (Tier 2): [0, 1, 2]
LOOKBACK   = 14                     # same history as the other models (optional ablation: 28)
NEG_RATE   = 0.01                   # same training sample as the other models
D_MODEL, N_LAYERS, DROPOUT = 64, 2, 0.1
CAMPAIGN_LOSS_WEIGHT = 0.1
COPY_HISTORY = True
EPOCHS, PATIENCE = 10, 2
BATCH, EVAL_BATCH, LR = 1024, 16384, 1e-3
CAMPAIGN_EVAL_MAX = 200_000
FORCE      = False
SEED       = 42

DATA_DIR = DRIVE_DIR / "processed" / DATASET
RES_DIR  = DRIVE_DIR / "results" / ("v2_history" if USE_HISTORY_FEATURES else "v1")
OUT_DIR  = RES_DIR / "itransformer" / DATASET
(OUT_DIR / "raw").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "models").mkdir(parents=True, exist_ok=True)
print("Data:", DATA_DIR, "\nResults:", OUT_DIR)

Mounted at /content/drive
Data: /content/drive/MyDrive/DATA612_data/processed/criteo_w12h 
Results: /content/drive/MyDrive/DATA612_data/results/v2_history/itransformer/criteo_w12h


## 2. Repo (shared metrics) and packages

In [2]:
REPO_URL = "https://github.com/Abhyudaya01/criteo-customer-journey-transformers.git"
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False)
elif IN_COLAB:
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))

need = [p for p, m in {"pyarrow": "pyarrow", "psutil": "psutil"}.items() if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)

import numpy as np, pandas as pd, torch, psutil
import torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import average_precision_score
from src.evaluation.metrics import conversion_metrics, best_f1_threshold, campaign_ranking_metrics

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda" and torch.cuda.is_bf16_supported()      # bfloat16 on A100 / L4
print(f"Device: {DEVICE} | bf16 autocast: {AMP} | RAM {psutil.virtual_memory().total / 1e9:.0f} GB | torch {torch.__version__}")

Device: cuda | bf16 autocast: True | RAM 179 GB | torch 2.11.0+cu130


## 3. Load the processed tables (identical to the other notebooks)

In [3]:
t0 = time.time()
meta = json.loads((DATA_DIR / "meta.json").read_text())
N_WINDOWS = meta["n_windows"]
N_CAMPAIGNS = meta["n_campaigns"]

WINDOW_HOURS = float(meta.get("window_hours", 12))
from src.data.io import shard_ids as data_shards
from src.data.features import build_position_features, feature_matrix, FEATURE_COLUMNS
HIST_COLS = [c for c in FEATURE_COLUMNS if c not in HIST_DROP] if USE_HISTORY_FEATURES else []
hist_keep = [FEATURE_COLUMNS.index(c) for c in HIST_COLS]
pos_parts, uw_parts, h_parts = [], [], []
for s in data_shards(DATA_DIR):                               # shard by shard: tables of a shard share users
    p = pd.read_parquet(DATA_DIR / "positions" / f"part-{s:03d}.parquet",
                        columns=["uid", "window", "label_window", "windows_since_active", "is_active",
                                 "y_conversion", "n_next_campaigns", "time_split", "user_group"])
    u = pd.read_parquet(DATA_DIR / "user_windows" / f"part-{s:03d}.parquet")
    if HIST_COLS:                                             # past-only (as-of merge on windows <= t)
        f = build_position_features(p[["uid", "window", "windows_since_active", "is_active"]], u, WINDOW_HOURS)
        h_parts.append(feature_matrix(f)[:, hist_keep].astype(np.float16))
        del f
    pos_parts.append(p); uw_parts.append(u)
pos = pd.concat(pos_parts, ignore_index=True)
uw = pd.concat(uw_parts, ignore_index=True)
H = np.concatenate(h_parts) if HIST_COLS else np.zeros((len(pos), 0), dtype=np.float16)
del pos_parts, uw_parts, h_parts
print(f"history features: {HIST_COLS if HIST_COLS else 'off'}")

# Contiguous user index and a dense (user, window) -> user_windows row lookup.
uids = np.unique(uw["uid"].to_numpy())
W = 1 << max(6, math.ceil(math.log2(N_WINDOWS)))           # window slots per user (64 for 12h)
uw_u = np.searchsorted(uids, uw["uid"].to_numpy()).astype(np.int64)
dense = np.full(len(uids) * W, len(uw), dtype=np.int32)     # len(uw) = "no record" (zero row)
dense[uw_u * W + uw["window"].to_numpy()] = np.arange(len(uw), dtype=np.int32)

# Per-window features (only past/present information; conversions keyed on their own timestamp).
FEATS = ["log_impressions", "log_clicks", "log_campaigns", "log_cost", "log_conversions", "is_active", "has_record"]
Fw = np.zeros((len(uw) + 1, len(FEATS)), dtype=np.float32)
Fw[:-1, 0] = np.log1p(uw["n_impressions"].to_numpy())
Fw[:-1, 1] = np.log1p(uw["n_clicks"].to_numpy())
Fw[:-1, 2] = np.log1p(uw["n_campaigns"].to_numpy())
Fw[:-1, 3] = np.log1p(uw["cost_sum"].to_numpy().clip(min=0))
Fw[:-1, 4] = np.log1p(uw["n_conversions"].to_numpy())
Fw[:-1, 5] = uw["is_active"].to_numpy()
Fw[:-1, 6] = 1.0

pos_u = np.searchsorted(uids, pos["uid"].to_numpy())
assert (uids[np.minimum(pos_u, len(uids) - 1)] == pos["uid"].to_numpy()).all(), "position user missing from user_windows"
pos_u = pos_u.astype(np.int64)
pos_w = pos["window"].to_numpy().astype(np.int64)
pos_wsa = pos["windows_since_active"].to_numpy().astype(np.float32)
y_all = pos["y_conversion"].to_numpy().astype(np.int8)
SPLIT_CODE = {"train": 0, "val": 1, "test": 2}
def split_codes(col):
    # Compact int8 codes (0 train, 1 val, 2 test); avoids 126M Python strings in memory.
    c = pos[col].astype("category")
    lut = np.array([SPLIT_CODE.get(str(k), -1) for k in c.cat.categories], dtype=np.int8)
    return lut[c.cat.codes.to_numpy()]
time_split = split_codes("time_split")
user_group = split_codes("user_group")
# Group per user (constant within a user), aligned with `uids`; -1 = user has no positions.
first = np.unique(pos_u, return_index=True)
uid_group_code = np.full(len(uids), -1, dtype=np.int8)
uid_group_code[first[0]] = user_group[first[1]]
pos = pos.drop(columns=["time_split", "user_group", "label_window", "window", "windows_since_active",
                        "is_active", "y_conversion"])
del uw_u, first

print(f"{len(pos):,} positions | {int(y_all.sum()):,} positives | {len(uids):,} users | "
      f"{len(uw):,} user-windows | loaded in {time.time() - t0:.0f}s")
if DATASET == "criteo_w12h":
    print("Expected (full 12h data): 125,971,114 positions and 231,004 positives")

history features: ['windows_since_active', 'is_active', 'cur_impressions', 'cur_clicks', 'cur_campaigns', 'cum_impressions', 'cum_clicks', 'cum_conversions', 'cum_active_windows', 'cum_cost', 'windows_since_first', 'windows_since_conversion', 'click_rate', 'slot_of_day']
125,971,114 positions | 231,004 positives | 6,142,256 users | 13,939,200 user-windows | loaded in 222s
Expected (full 12h data): 125,971,114 positions and 231,004 positives


## 4. Features, splits and training sample (identical to the other notebooks)

In [4]:
LAGS = np.arange(LOOKBACK - 1, -1, -1)                       # oldest ... newest (t)

def split_masks(protocol):
    if protocol == "chronological":
        return {s: time_split == c for s, c in SPLIT_CODE.items()}
    if protocol == "user_disjoint":                          # unseen users, all windows (run_baselines.py)
        return {s: user_group == c for s, c in SPLIT_CODE.items()}
    if protocol == "user_disjoint_future":                   # unseen users and future windows
        return {s: (time_split == c) & (user_group == c) for s, c in SPLIT_CODE.items()}
    raise ValueError(protocol)

rng = np.random.default_rng(SEED)
keep_neg = rng.random(len(pos)) < NEG_RATE                   # one fixed negative sample for all models

# Standardise the history features with statistics from (sampled) chronological-training rows only.
_ref = np.flatnonzero((time_split == SPLIT_CODE["train"]) & keep_neg)
H_MU = H[_ref].astype(np.float32).mean(0) if H.shape[1] else np.zeros(0, np.float32)
H_SD = H[_ref].astype(np.float32).std(0) + 1e-6 if H.shape[1] else np.ones(0, np.float32)

def static_features(idx):
    # Per-position features: recency, calendar, plus the standardised full-history features.
    t = pos_w[idx]
    base = np.stack([pos_wsa[idx] / LOOKBACK, (t % 2).astype(np.float32),
                     np.sin(2 * np.pi * (t % 14) / 14).astype(np.float32),
                     np.cos(2 * np.pi * (t % 14) / 14).astype(np.float32)], axis=1)
    if not H.shape[1]:
        return base
    return np.concatenate([base, (H[idx].astype(np.float32) - H_MU) / H_SD], axis=1).astype(np.float32)
N_STATIC = 4 + H.shape[1]

def build_features(idx):
    # (n, LOOKBACK, F) window history built only from windows <= t, plus (n, N_STATIC) static features.
    u, t = pos_u[idx], pos_w[idx]
    w = t[:, None] - LAGS[None, :]
    rows = dense[u[:, None] * W + np.clip(w, 0, None)]
    rows[w < 0] = len(uw)
    return Fw[rows], static_features(idx)

def train_sample(masks):
    return np.flatnonzero(masks["train"] & ((y_all == 1) | keep_neg))

def to_true_prob(p, r=NEG_RATE):
    # Undo negative subsampling: true odds = sampled odds * r.
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return p * r / (p * r + 1 - p)

for prot in PROTOCOLS:
    m = split_masks(prot)
    tr = train_sample(m)
    print(f"{prot:<22} train sample {len(tr):,} ({int(y_all[tr].sum()):,} pos) | "
          f"val {int(m['val'].sum()):,} ({int(y_all[m['val']].sum()):,} pos) | "
          f"test {int(m['test'].sum()):,} ({int(y_all[m['test']].sum()):,} pos)")

chronological          train sample 1,010,876 (158,834 pos) | val 20,421,186 (34,430 pos) | test 20,121,228 (37,740 pos)
user_disjoint          train sample 1,042,399 (162,013 pos) | val 18,882,244 (34,440 pos) | test 18,878,625 (34,551 pos)


## 5. Batches: activity series, campaign history and labels (identical to the Lu & Kannan notebook)

In [5]:
t0 = time.time()
wc = pd.read_parquet(DATA_DIR / "window_campaigns", columns=["uid", "window", "campaign_idx", "n_impressions"])
wc_row = dense[np.searchsorted(uids, wc["uid"].to_numpy()).astype(np.int64) * W + wc["window"].to_numpy()]
assert (wc_row < len(uw)).all(), "window_campaigns row without a user_windows row"
order = np.argsort(wc_row, kind="stable")
CAMP_IDX = wc["campaign_idx"].to_numpy()[order].astype(np.int64)
CAMP_WT  = np.log1p(wc["n_impressions"].to_numpy()[order]).astype(np.float32)
CAMP_OFF = np.searchsorted(wc_row[order], np.arange(len(uw) + 2)).astype(np.int64)   # row len(uw) = empty
del wc, wc_row, order

ncl = pd.read_parquet(DATA_DIR / "next_campaign_labels", columns=["uid", "window", "campaign_idx"])
ncl_key = np.searchsorted(uids, ncl["uid"].to_numpy()).astype(np.int64) * W + ncl["window"].to_numpy()
ncl_order = np.argsort(ncl_key, kind="stable")
NCL_KEY, NCL_CAMP = ncl_key[ncl_order], ncl["campaign_idx"].to_numpy()[ncl_order].astype(np.int64)
del ncl, ncl_key, ncl_order
print(f"Campaign tokens and labels ready in {time.time() - t0:.0f}s")

def campaign_bags(rows):
    # Flattened EmbeddingBag inputs for an array of user_windows rows.
    starts, ends = CAMP_OFF[rows], CAMP_OFF[rows + 1]
    lens = ends - starts
    offsets = np.concatenate([[0], np.cumsum(lens)[:-1]]).astype(np.int64)
    idx = np.repeat(starts - offsets, lens) + np.arange(int(lens.sum()))
    return CAMP_IDX[idx], CAMP_WT[idx], offsets, lens

def next_campaigns(idx):
    # (row_in_batch, campaign) pairs for campaigns in window t+1 of each position.
    keys = pos_u[idx] * W + pos_w[idx]
    lo, hi = np.searchsorted(NCL_KEY, keys, "left"), np.searchsorted(NCL_KEY, keys, "right")
    lens = hi - lo
    rows = np.repeat(np.arange(len(idx)), lens)
    j = np.repeat(lo - np.concatenate([[0], np.cumsum(lens)[:-1]]), lens) + np.arange(int(lens.sum()))
    return rows, NCL_CAMP[j]

def seen_matrix(idx, chunk=50_000):
    # True where the user saw the campaign in any window <= t ("not new"), for the new-campaign metrics.
    seen = np.zeros((len(idx), N_CAMPAIGNS), dtype=bool)
    wins = np.arange(N_WINDOWS)
    for s in range(0, len(idx), chunk):
        j = idx[s:s + chunk]
        rows = dense[pos_u[j][:, None] * W + wins[None, :]]
        rows[wins[None, :] > pos_w[j][:, None]] = len(uw)
        camp, _, _, lens = campaign_bags(rows.reshape(-1))
        seen[s + np.repeat(np.arange(rows.size), lens) // N_WINDOWS, camp] = True
    return seen

def ranking_all_and_new(y_true, seen, scores):
    out = {k: v for k, v in campaign_ranking_metrics(y_true, scores, ks=(1, 5, 10)).items()
           if k != "frac_positions_nonempty"}
    y_new = (y_true.astype(bool) & ~seen).astype(np.int8)
    keep = y_new.sum(1) > 0
    new = campaign_ranking_metrics(y_new[keep], scores[keep], ks=(1, 5, 10)) if keep.any() else {}
    out.update({f"new_{k}": v for k, v in new.items() if k != "frac_positions_nonempty"})
    out["frac_positions_with_new_campaign"] = float(keep.mean())
    out["frac_next_campaigns_new"] = float(y_new.sum() / max(y_true.sum(), 1))
    return out

def make_batch(idx, with_labels=False):
    # Tensors for one batch of positions: numeric tokens, campaign bags, static, padding mask, user index.
    u, t = pos_u[idx], pos_w[idx]
    w = t[:, None] - LAGS[None, :]                                   # (n, L) absolute windows, oldest first
    rows = dense[u[:, None] * W + np.clip(w, 0, None)]
    rows[w < 0] = len(uw)
    cal = np.stack([(w % 2), np.sin(2 * np.pi * (w % 14) / 14), np.cos(2 * np.pi * (w % 14) / 14)], -1)
    Xnum = np.concatenate([Fw[rows], cal.astype(np.float32)], axis=-1)
    S = static_features(idx)                                          # same static features as the baselines
    camp, wt, off, lens = campaign_bags(rows.reshape(-1))
    # History score per (position, campaign): recency-weighted log impressions over the lookback (windows <= t).
    elem_row = np.repeat(np.arange(rows.size), lens)
    elem_w = wt * (0.8 ** LAGS[elem_row % LOOKBACK]).astype(np.float32)
    b = {"x": torch.from_numpy(Xnum), "camp": torch.from_numpy(camp), "wt": torch.from_numpy(wt),
         "off": torch.from_numpy(off), "s": torch.from_numpy(S), "pad": torch.from_numpy(w < 0),
         "u": torch.from_numpy(u), "h_row": torch.from_numpy(elem_row // LOOKBACK),
         "h_w": torch.from_numpy(elem_w)}
    if with_labels:
        b["y"] = torch.from_numpy(y_all[idx].astype(np.float32))
        r, c = next_campaigns(idx)
        b["yc_r"], b["yc_c"] = torch.from_numpy(r), torch.from_numpy(c)
    return {k: v.to(DEVICE, non_blocking=True) for k, v in b.items()}

Campaign tokens and labels ready in 18s


## 6. Model: iTransformer

In [6]:
N_ACTIVITY = Fw.shape[1]             # the 7 activity series

def top_campaigns(tr, k=TOP_CAMPAIGNS, max_rows=1_000_000):
    # Most frequent campaigns (by log-impressions) in the input windows of the training sample only.
    j = tr if len(tr) <= max_rows else np.random.default_rng(SEED).choice(tr, size=max_rows, replace=False)
    w = pos_w[j][:, None] - LAGS[None, :]
    rows = dense[pos_u[j][:, None] * W + np.clip(w, 0, None)]
    rows[w < 0] = len(uw)
    camp, wt, _, _ = campaign_bags(rows.reshape(-1))
    counts = np.bincount(camp, weights=wt, minlength=N_CAMPAIGNS)
    return np.argsort(-counts, kind="stable")[:min(k, N_CAMPAIGNS)]

class ITransformer(nn.Module):
    def __init__(self, top_idx, n_static, n_camp, n_heads, d=D_MODEL, layers=N_LAYERS, dropout=DROPOUT, lookback=LOOKBACK):
        super().__init__()
        self.k, self.n_heads, self.L = len(top_idx), n_heads, lookback
        tok = torch.full((n_camp,), self.k, dtype=torch.long)                 # campaign -> token ("other" = k)
        tok[torch.as_tensor(top_idx, dtype=torch.long)] = torch.arange(self.k)
        self.register_buffer("camp_tok", tok)
        self.register_buffer("top_idx", torch.as_tensor(top_idx, dtype=torch.long))
        self.n_var = self.k + 1 + N_ACTIVITY
        self.var_proj = nn.Linear(lookback, d)                                # inverted embedding: series -> token
        self.var_emb = nn.Parameter(torch.randn(self.n_var, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, n_heads, dim_feedforward=4 * d, dropout=dropout,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.conv_head = nn.Sequential(nn.Linear(2 * d + n_static, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))
        self.camp_head = nn.Sequential(nn.Linear(2 * d, 128), nn.ReLU(), nn.Linear(128, n_camp))
        self.tok_head = nn.Linear(d, 1)                                       # each top campaign scores itself
        self.copy_scale = nn.Parameter(torch.tensor(1.0)) if COPY_HISTORY else None

    def campaign_series(self, b, B):
        # (B, L, k+1): per-window log-impressions of each top campaign and of all other campaigns together.
        lens = torch.diff(b["off"], append=torch.tensor([b["camp"].numel()], device=b["off"].device))
        flat_row = torch.repeat_interleave(torch.arange(B * self.L, device=lens.device), lens)
        series = torch.zeros(B * self.L, self.k + 1, device=lens.device)
        series.index_put_((flat_row, self.camp_tok[b["camp"]]), b["wt"], accumulate=True)
        return series.view(B, self.L, self.k + 1)

    def forward(self, b):
        x = b["x"][..., :N_ACTIVITY]                                          # (B, L, 7)
        B = x.shape[0]
        v = torch.cat([self.campaign_series(b, B), x], dim=-1)                # (B, L, V)
        z = self.var_proj(v.transpose(1, 2)) + self.var_emb                   # (B, V, d): one token per variable
        # Chunked: CUDA attention fails when sequences x heads exceeds 65,535 in one call.
        chunk = max(1, 32_768 // self.n_heads)
        z = self.norm(torch.cat([self.encoder(z[i:i + chunk]) for i in range(0, B, chunk)]))
        pooled = torch.cat([z.mean(1), z.max(1).values], dim=1)
        conv_logit = self.conv_head(torch.cat([pooled, b["s"]], 1)).squeeze(-1)
        camp_logits = self.camp_head(pooled).float()
        tok_scores = self.tok_head(z[:, :self.k]).squeeze(-1).float()         # (B, k)
        camp_logits = camp_logits.index_add(1, self.top_idx, tok_scores)
        if self.copy_scale is not None:
            hist = torch.zeros_like(camp_logits)
            hist.index_put_((b["h_row"], b["camp"]), b["h_w"], accumulate=True)
            camp_logits = camp_logits + self.copy_scale * hist
        return conv_logit, camp_logits, None

## 7. Training and prediction (same procedure as the Lu & Kannan notebook)

In [7]:
def autocast():
    return torch.autocast("cuda", dtype=torch.bfloat16) if AMP else torch.autocast("cpu", enabled=False)

def predict(model, idx, campaign=False):
    model.eval()
    conv, camp = np.empty(len(idx), dtype=np.float64), []
    with torch.no_grad():
        for s in range(0, len(idx), EVAL_BATCH):
            with autocast():
                lc, lk, _ = model(make_batch(idx[s:s + EVAL_BATCH]))
            conv[s:s + EVAL_BATCH] = torch.sigmoid(lc.float()).cpu().numpy()
            if campaign:
                camp.append(torch.sigmoid(lk.float()).cpu().numpy())
    return to_true_prob(conv), (np.concatenate(camp) if campaign else None)

def train_model(tr, val_idx, n_heads, seed, tag):
    torch.manual_seed(seed); np.random.seed(seed)
    top_idx = top_campaigns(tr)                                       # from this protocol's training sample only
    model = ITransformer(top_idx, n_static=N_STATIC, n_camp=N_CAMPAIGNS, n_heads=n_heads).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    vs = val_idx[(y_all[val_idx] == 1) | keep_neg[val_idx]]
    rs = np.random.default_rng(seed)
    best, best_state, bad = -1.0, None, 0
    for ep in range(EPOCHS):
        model.train()
        perm = tr[rs.permutation(len(tr))]
        t0, tot = time.time(), 0.0
        for s in range(0, len(perm), BATCH):
            b = make_batch(perm[s:s + BATCH], with_labels=True)
            with autocast():
                lc, lk, _ = model(b)
            yc = torch.zeros(lk.shape, device=DEVICE)
            yc[b["yc_r"], b["yc_c"]] = 1.0
            loss = F.binary_cross_entropy_with_logits(lc.float(), b["y"]) + \
                   CAMPAIGN_LOSS_WEIGHT * F.binary_cross_entropy_with_logits(lk.float(), yc, reduction="sum") / len(yc)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item() * len(yc)
        p_vs, _ = predict(model, vs)
        ap = average_precision_score(y_all[vs], p_vs)
        print(f"    epoch {ep + 1}: loss {tot / len(perm):.4f} | val-sample PR-AUC {ap:.4f} | {time.time() - t0:.0f}s")
        if ap > best:
            best, bad = ap, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    torch.save(best_state, OUT_DIR / "models" / f"{tag}.pt")
    return model

## 8. Run: conversion + campaign evaluation for each setting

In [8]:
def campaign_sample(m):
    cand = np.flatnonzero(m["test"] & (pos["n_next_campaigns"].to_numpy() > 0))
    rs = np.random.default_rng(SEED)
    return np.sort(rs.choice(cand, size=min(CAMPAIGN_EVAL_MAX, len(cand)), replace=False)), len(cand)

def run(protocol, n_heads, seed):
    name = f"itransformer_top{TOP_CAMPAIGNS}" + (f"_L{LOOKBACK}" if LOOKBACK != 14 else "")
    tag = f"{protocol}_{name}_h{n_heads}_seed{seed}"
    path = OUT_DIR / "raw" / f"{tag}.json"
    if path.exists() and not FORCE:
        print(f"  {tag}: already done (skipping)")
        return json.loads(path.read_text())
    print(f"  {tag}: training")
    m = split_masks(protocol)
    tr = train_sample(m)
    val_idx, test_idx = np.flatnonzero(m["val"]), np.flatnonzero(m["test"])
    t0 = time.time()
    model = train_model(tr, val_idx, n_heads, seed, tag)
    fit_s = time.time() - t0
    p_val, _ = predict(model, val_idx)
    thr, _ = best_f1_threshold(y_all[val_idx], p_val)
    p_test, _ = predict(model, test_idx)
    res = {"task": "conversion", "protocol": protocol, "model": name, "top_campaigns": TOP_CAMPAIGNS,
           "heads": n_heads, "lookback": LOOKBACK, "seed": seed,
           "val_pr_auc": float(average_precision_score(y_all[val_idx], p_val)),
           **conversion_metrics(y_all[test_idx], p_test, threshold=thr),
           "f1_threshold_from_val": thr, "n_test": int(len(test_idx)), "n_test_pos": int(y_all[test_idx].sum()),
           "n_params": int(sum(p.numel() for p in model.parameters())), "fit_seconds": round(fit_s, 1)}
    cidx, _ = campaign_sample(m)
    _, scores = predict(model, cidx, campaign=True)
    y_true = np.zeros(scores.shape, dtype=np.int8)
    r, c = next_campaigns(cidx)
    y_true[r, c] = 1
    res["campaign"] = ranking_all_and_new(y_true, seen_matrix(cidx), scores)
    res["total_seconds"] = round(time.time() - t0, 1)
    path.write_text(json.dumps(res, indent=2))
    print(f"  {tag}: test PR-AUC {res['pr_auc']:.4f} | ROC-AUC {res['roc_auc']:.4f} | "
          f"campaign Recall@10 {res['campaign']['recall_at_10']:.4f} | "
          f"new-campaign Recall@10 {res['campaign'].get('new_recall_at_10', float('nan')):.4f} | {res['total_seconds']:.0f}s")
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return res

results = []
for prot in PROTOCOLS:
    print(f"\n=== {prot} ===")
    for h in HEADS:
        for seed in SEEDS:
            results.append(run(prot, h, seed))


=== chronological ===
  chronological_itransformer_top100_h4_seed0: training
    epoch 1: loss 0.9180 | val-sample PR-AUC 0.5503 | 21s
    epoch 2: loss 0.3765 | val-sample PR-AUC 0.5572 | 20s
    epoch 3: loss 0.3722 | val-sample PR-AUC 0.5614 | 20s
    epoch 4: loss 0.3692 | val-sample PR-AUC 0.5637 | 20s
    epoch 5: loss 0.3673 | val-sample PR-AUC 0.5649 | 20s
    epoch 6: loss 0.3654 | val-sample PR-AUC 0.5687 | 20s
    epoch 7: loss 0.3642 | val-sample PR-AUC 0.5677 | 20s
    epoch 8: loss 0.3631 | val-sample PR-AUC 0.5685 | 20s
  chronological_itransformer_top100_h4_seed0: test PR-AUC 0.0292 | ROC-AUC 0.8637 | campaign Recall@10 0.7117 | new-campaign Recall@10 0.0528 | 428s

=== user_disjoint ===
  user_disjoint_itransformer_top100_h4_seed0: training
    epoch 1: loss 0.8984 | val-sample PR-AUC 0.5696 | 20s
    epoch 2: loss 0.3735 | val-sample PR-AUC 0.5765 | 20s
    epoch 3: loss 0.3693 | val-sample PR-AUC 0.5795 | 20s
    epoch 4: loss 0.3664 | val-sample PR-AUC 0.5813 | 20s

## 9. Comparison with all models so far (saved to Drive)

In [9]:
# Every run saved so far (including earlier ablation settings), not only this session's runs.
results = [json.loads(f.read_text()) for f in sorted((OUT_DIR / "raw").glob("*.json"))]
it = pd.DataFrame(results)
conv_cols = ["protocol", "model", "seed", "pr_auc", "roc_auc", "f1", "balanced_accuracy", "binary_cross_entropy",
             "brier", "expected_calibration_error", "top_decile_conversion_rate", "top_decile_lift",
             "top_decile_gain", "base_rate", "val_pr_auc", "n_test", "n_test_pos", "fit_seconds"]
it[conv_cols + ["top_campaigns", "heads", "lookback", "n_params"]].to_csv(OUT_DIR / "conversion_itransformer.csv", index=False)
camp_rows = [{"protocol": r["protocol"], "model": r["model"], "seed": r["seed"], **r["campaign"]} for r in results]
pd.DataFrame(camp_rows).to_csv(OUT_DIR / "campaign_itransformer.csv", index=False)

conv_files = [RES_DIR / "baselines" / DATASET / "conversion_baselines.csv",
              RES_DIR / "lu_kannan" / DATASET / "conversion_lu_kannan.csv",
              RES_DIR / "patchtst" / DATASET / "conversion_patchtst.csv"]
camp_files = [RES_DIR / "baselines" / DATASET / "campaign_baselines.csv",
              RES_DIR / "lu_kannan" / DATASET / "campaign_lu_kannan.csv",
              RES_DIR / "patchtst" / DATASET / "campaign_patchtst.csv"]
conv = pd.concat([pd.read_csv(f) for f in conv_files if f.exists()] + [it[conv_cols]], ignore_index=True)
camp = pd.concat([pd.read_csv(f) for f in camp_files if f.exists()] + [pd.DataFrame(camp_rows)], ignore_index=True)

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print("Next-window conversion (test set):")
display(conv[["protocol", "model", "pr_auc", "roc_auc", "top_decile_lift", "top_decile_gain", "expected_calibration_error"]]
        .sort_values(["protocol", "pr_auc"], ascending=[True, False]).reset_index(drop=True))
print("\nNext-window campaigns (test set, positions with a non-empty next window):")
display(camp[["protocol", "model", "recall_at_10", "hit_rate_at_10", "ndcg_at_10", "new_recall_at_10", "new_ndcg_at_10"]]
        .sort_values(["protocol", "recall_at_10"], ascending=[True, False]).reset_index(drop=True))

import platform
(OUT_DIR / "run_config.json").write_text(json.dumps({
    "python": platform.python_version(), "torch": torch.__version__, "device": DEVICE, "bf16": AMP,
    "dataset": DATASET, "lookback": LOOKBACK, "neg_rate": NEG_RATE, "d_model": D_MODEL, "layers": N_LAYERS,
    "dropout": DROPOUT, "campaign_loss_weight": CAMPAIGN_LOSS_WEIGHT, "copy_history": COPY_HISTORY,
    "batch": BATCH, "lr": LR, "top_campaigns": TOP_CAMPAIGNS, "heads": HEADS, "seeds": SEEDS, "history_features": HIST_COLS,
    "protocols": PROTOCOLS}, indent=2))
print("\nSaved to:", OUT_DIR)

Next-window conversion (test set):


,protocol,model,pr_auc,roc_auc,top_decile_lift,top_decile_gain,expected_calibration_error
0,chronological,itransformer_top100,0.0292,0.8637,5.8895,0.5890,0.0002
1,chronological,lstm,0.0264,0.8489,5.4971,0.5497,0.0006
2,chronological,gbdt,0.0256,0.8491,5.4992,0.5499,0.0003
3,chronological,lu_kannan_per_user_h4,0.0255,0.8631,5.8415,0.5842,0.0007
4,chronological,patchtst_p2s2,0.0252,0.8465,5.4375,0.5437,0.0006
5,chronological,logreg,0.0193,0.8200,4.8670,0.4867,0.0009
6,chronological,recency,0.0119,0.7537,4.2655,0.4266,0.0009
7,chronological,hmm,0.0055,0.5656,2.1161,0.2116,0.0013
8,chronological,prior,0.0019,0.5000,1.0074,0.1007,0.0000
9,user_disjoint,itransformer_top100,0.0298,0.8654,5.9851,0.5985,0.0001



Next-window campaigns (test set, positions with a non-empty next window):


,protocol,model,recall_at_10,hit_rate_at_10,ndcg_at_10,new_recall_at_10,new_ndcg_at_10
0,chronological,user_history,0.8858,0.8954,0.8337,0.1222,0.0488
1,chronological,user_recency_7d,0.8306,0.8441,0.8000,0.1275,0.0526
2,chronological,itransformer_top100,0.7117,0.7251,0.6773,0.0528,0.0219
3,chronological,lu_kannan_per_user_h4,0.6944,0.7084,0.6131,0.1326,0.0615
4,chronological,patchtst_p2s2,0.5915,0.6043,0.5302,0.0632,0.0259
5,chronological,popularity,0.1863,0.1919,0.0947,0.1372,0.0680
6,user_disjoint,user_history,0.8601,0.8732,0.8192,0.1386,0.0556
7,user_disjoint,user_recency_7d,0.8247,0.8402,0.7947,0.1409,0.0575
8,user_disjoint,itransformer_top100,0.7377,0.7526,0.7032,0.0850,0.0353
9,user_disjoint,patchtst_p2s2,0.6271,0.6424,0.5734,0.0998,0.0400



Saved to: /content/drive/MyDrive/DATA612_data/results/v2_history/itransformer/criteo_w12h


## Notes for the report
- **Top campaigns** are chosen from each protocol's training sample only (no validation/test information).
- **Variable-identity embedding**: added because many sparse campaign series are all-zero and would otherwise give
  identical tokens; the original iTransformer relies on series content alone.
- **Ablations (optional):** `TOP_CAMPAIGNS = 50` or `200`; `HEADS = [1, 2, 4]`; `SEEDS = [0, 1, 2]` for the final table.
- LoRA vs full fine-tuning on iTransformer is Tier 3 (proposal); not included here.